# Earth Engine shard (Colab)

Runs one quarter of the Earth Engine extraction. **Change only `PROJECT_ID` and `SHARD` in the next cell**, then Runtime > Run all.

- `PROJECT_ID`: your Google Cloud project registered for Earth Engine (https://code.earthengine.google.com/register)
- `SHARD`: 0, 1, 2 or 3 (one per person, no repeats)

Finished batches are saved to your Google Drive, so if Colab disconnects, just run all again and it resumes.

In [ ]:
PROJECT_ID = "doppelgrund"   # <- change me
SHARD = 0                    # <- change me: 0, 1, 2 or 3

N_SHARDS = 4                 # leave alone: the same on every machine
STRIDE = 20                  # leave alone: the same on every machine (use every 20th hex = fast partial run; 1 = everything)
THREADS = 4                  # lower to 2 if Earth Engine reports quota errors

In [ ]:
# Drive keeps the finished batches if the Colab session dies
import sys
from google.colab import drive

assert sys.version_info >= (3, 12), f"needs Python 3.12+, this Colab has {sys.version}"
drive.mount("/content/drive")
OUT = f"/content/drive/MyDrive/DoppelGrund_ee/shard{SHARD}"

In [ ]:
# code from dev-branch (re-run safe: pulls if already cloned)
import os

if os.path.isdir("/content/DoppelGrund"):
    !git -C /content/DoppelGrund pull origin dev-branch
else:
    !git clone -b dev-branch https://github.com/mackerel7111/DoppelGrund.git /content/DoppelGrund
%cd /content/DoppelGrund
!pip install -q -e "pipeline[ee]"
!git log --oneline -1

In [ ]:
# one-time Google sign-in for Earth Engine: open the link, sign in, paste the code back
import ee

ee.Authenticate(auth_mode="notebook")

In [ ]:
# the extraction: prints "N batches to run", then "batch k/818 done in Xs" per batch. It ends when the cell stops with no error.
!python -m doppelgrund.real --ee-project {PROJECT_ID} --shard {SHARD}/{N_SHARDS} --threads {THREADS} --stride {STRIDE} --out "{OUT}"

In [ ]:
# done check: expect about 10 files per shard at STRIDE=20 (41 batches in total), then send shard zip to whoever merges
!ls "{OUT}/ee_batches" | wc -l
!cd "{OUT}/ee_batches" && zip -q ../shard_{SHARD}.zip *.parquet
print(f"zip is in your Drive: {OUT}/shard_{SHARD}.zip")